ERPs Analysis Procedure

1. Load raw EEG file for 1 subject

2. reprocess data:
  - Filtering
  - Artifact removal
3. Find event markers ("Wd2E" & "Wd2N")

4. Create epochs
  - Time-locked to events|
  - Baseline correction

5. Save clean epochs for 1 subject

6. Loop through all participants and collect clean epochs separately for "Wd2E" and "Wd2N" for each participant

7. Average within each subject
   - Subject-level ERP (all E trials)
   - Subject-level ERP (all N trials)

8. Average across subjects  → Grand average ERPs
   - Grand average ERP (all E trials)
   - Grand average ERP (all N trials)



**Independent Component Analysis** is a statistical method that separates multichannel signals (like EEG) into **independent sources**, under the assumption that:

* The observed EEG signals are **linear mixtures** of underlying sources.
* These sources (e.g., brain activity, eye blinks, muscle activity) are **statistically independent** and **non-Gaussian**.

In EEG:

> ICA helps **unmix** signals into components like eye blinks, heartbeats, muscle activity, and brain signals.

## ICA in EEG: Two Main Steps

### 1. **Automatic Component Selection**

Use statistical correlation with artifacts like eye blinks or heartbeat.

#### Eye blink detection:

```python
eog_inds, scores = ica.find_bads_eog(raw)
ica.exclude = eog_inds
```

* Finds components whose time courses strongly correlate with **EOG channels** (or frontopolar EEG channels).
* You can visualize:

  ```python
  ica.plot_scores(scores)          # correlation values
  ica.plot_properties(raw, picks=eog_inds)
  ```

#### ECG artifact (if available):

```python
ecg_inds, scores = ica.find_bads_ecg(raw)
ica.exclude.extend(ecg_inds)
```

---

### 2. **Manual Component Selection**

If automatic selection is imperfect, you can inspect components manually:

```python
ica.plot_components()             # Topographic map of each IC
ica.plot_sources(raw)            # Time course of each IC
```

Then:

```python
ica.exclude = [0, 2, 4]           # Just an example — use your judgment
```

Look for:

* **Eye blinks**: strong frontal topography, sharp peaks in time series
* **Muscle artifacts**: high-frequency noise
* **Heartbeats**: regular peaks across time


In [ ]:
# =============================================
# 0. SETUP
# =============================================
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet
import mne
import os
import numpy as np
from scipy.signal import hilbert
import matplotlib.pyplot as plt

In [ ]:
# =============================================
# 1. PARAMETERS
# =============================================
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEG Data' # main data directory
output_dir = os.path.join(root_dir, 'Group_Level_Results') #saving processed results
os.makedirs(output_dir, exist_ok=True)

event_id = {'Wd2N': 1, 'Wd2E': 2}  # Event codes “Wd2N” = slowed context, “Wd2E” = normal contex
tmin, tmax = -0.1, 0.8  # Epoch window: from -100 ms to 800 ms around event onset.
baseline = (tmin, 0)     # Baseline correction

In [ ]:
# =============================================
# 2. PROCESSING FUNCTIONS
# =============================================
def process_file(raw_file):
    """Process single EEG .set file."""
    try:
        print(f"📥 Processing file: {raw_file}")
        # Load one raw EEG data
        raw = mne.io.read_raw_eeglab(raw_file, preload=True)
        raw.filter(0.1, None)      # High-pass filter above 0.1 Hz to remove slow drifts
        raw.notch_filter(60)       # Notch filter at 60 Hz to remove line noise

        # --------------------------------------------------
        # Artifact removal using ICA
        # --------------------------------------------------
        ica = mne.preprocessing.ICA(n_components=15, random_state=97, max_iter='auto')
        # extract 15 statistically independent sources
        # random_state=97: Ensures reproducibility.
        # max_iter='auto': Let MNE decide when to stop iterating
        ica.fit(raw)  # Fit ICA to raw data

        # --------------------------------------------------
        # AUTOMATIC: Detect and mark eye blink components
        # --------------------------------------------------
        eog_inds, eog_scores = ica.find_bads_eog(raw)  # detect components that match typical eye-blink activity
        print(f"Auto-detected EOG-related components (likely blinks/movements): {eog_inds}")
        ica.exclude = eog_inds  # mark components for removal

        # --------------------------------------------------
        # Save ICA plots for manual inspection
        # --------------------------------------------------
        participant_path = os.path.dirname(raw_file)
        participant = os.path.basename(participant_path)

        try:
            fig_components = ica.plot_components(show=False)
            fig_path = os.path.join(participant_path, f"{participant}_ica_components.png")
            if fig_components:
                fig_components[0].savefig(fig_path)
                print(f"Saved ICA topographies to: {fig_path}")
            else:
                print("❌ ICA component plot is empty — skipping save.")
        except Exception as e:
            print(f"❌ Error saving ICA topographies: {e}")

        try:
            fig_sources = ica.plot_sources(raw, show=False)
            ts_path = os.path.join(participant_path, f"{participant}_ica_sources.png")
            fig_sources.savefig(ts_path)
            print(f"Saved ICA sources (time series) to: {ts_path}")
        except Exception as e:
            print(f"❌ Error saving ICA source time series: {e}")

        # --------------------------------------------------
        # MANUAL INPUT: You inspect and enter bad components
        # --------------------------------------------------
        # manual_input = input(
            #f"\n✋ Enter additional component numbers to exclude for {participant} (comma-separated, e.g. 0,2,4):\n"
        #)
        #if manual_input.strip():
            # exclude_list = [int(i.strip()) for i in manual_input.split(",")]
            # ica.exclude.extend(exclude_list)  # Add to previously auto-excluded EOG components
        # print(f"Excluding components: {ica.exclude}")

        # Save excluded components list
        #exclusion_path = os.path.join(participant_path, f"{participant}_excluded_components.txt")
        #with open(exclusion_path, "w") as f:
            #f.write(f"Excluded ICA components: {ica.exclude}\n")
        #print(f"Saved exclusion list to: {exclusion_path}")

        # --------------------------------------------------
        # Apply ICA: Clean raw data
        # --------------------------------------------------
        raw_clean = ica.apply(raw)

        # --------------------------------------------------
        # Extract events markers (“Wd2E”, “Wd2N”) from the file’s annotations
        # --------------------------------------------------
        events, event_dict = mne.events_from_annotations(raw_clean)

        # Creates epochs around event markers using event_id dict to find “Wd2E” & “Wd2N”
        epochs = mne.Epochs(raw_clean, events, event_id=event_id, tmin=tmin, tmax=tmax,
                            baseline=baseline, preload=True, reject_by_annotation=True)

        # Baseline correction for DC offsets by subtracting mean of baseline period
        epochs.apply_baseline(baseline)
        print(f"✅ Finished processing {raw_file}")
        return epochs

    except Exception as e:
        print(f"❌ Error processing {raw_file}: {str(e)}")
        return None


In [ ]:
# =============================================
# 3. MAIN PROCESSING LOOP
# =============================================
all_epochs = {'Wd2E': [], 'Wd2N': []}

participants = sorted([p for p in os.listdir(root_dir)
                       if os.path.isdir(os.path.join(root_dir, p))])

for participant in participants:
    participant_path = os.path.join(root_dir, participant)
    epoched_file = os.path.join(participant_path, f"{participant}_epochs-epo.fif")

    # Temporarily force reprocessing — skip loading .fif
    # if os.path.exists(epoched_file):
    #     epochs = mne.read_epochs(epoched_file)
    #     print(f"Loaded preprocessed epochs for {participant}")
    # else:

    # Find .set file
    set_files = [f for f in os.listdir(participant_path) if f.endswith('.set')]
    if not set_files:
        print(f"No .set file found for {participant}")
        continue

    # Process .set file
    raw_file = os.path.join(participant_path, set_files[0])
    epochs = process_file(raw_file)

    if epochs:
        epochs.save(epoched_file, overwrite=True)
    else:
        continue

    # Separate and store for each trial condition
    for cond in event_id:
        if cond in epochs.event_id:
            cond_epochs = epochs[cond]
            all_epochs[cond].append(cond_epochs)


In [ ]:
# =============================================
# 4. GROUP-LEVEL ANALYSIS
# =============================================

# Filter out any empty or invalid entries before averaging
evokeds_E = [ep.average() for ep in all_epochs['Wd2E'] if ep is not None and len(ep) > 0]
evokeds_N = [ep.average() for ep in all_epochs['Wd2N'] if ep is not None and len(ep) > 0]

# Check if lists are still non-empty
if evokeds_E:
    erp_E = mne.grand_average(evokeds_E)
else:
    print("Warning: No valid 'Wd2E' epochs found for grand average.")
    erp_E = None

if evokeds_N:
    erp_N = mne.grand_average(evokeds_N)
else:
    print("Warning: No valid 'Wd2N' epochs found for grand average.")
    erp_N = None

In [ ]:
# =============================================
# 5. PLOTTIN ERPs (9 REGIONS GRID)
# =============================================

# Define 9 scalp regions with anatomically relevant channels
regions = {
    'Left Anterior':     ['E9', 'E2', 'E3', 'E4', 'E5'],
    'Medial Anterior':   ['E6', 'E11', 'E12', 'E13', 'E16'],
    'Right Anterior':    ['E117', 'E118', 'E119', 'E120', 'E123'],
    'Left Central':      ['E45', 'E46', 'E41', 'E36', 'E42'],
    'Medial Central':    ['E13', 'E31', 'E80', 'E112', 'E7'],
    'Right Central':     ['E105', 'E111', 'E104', 'E108', 'E102'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E59', 'E60'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77'],
    'Right Posterior':   ['E85', 'E86', 'E91', 'E97', 'E101']
}

fig, axs = plt.subplots(3, 3, figsize=(15, 10), sharex=True, sharey=True)

for i, (region, ch_list) in enumerate(regions.items()):
    ax = axs[i // 3, i % 3]

    # Keep only the channels that exist in the data and aren’t bad
    good_ch_list = [ch for ch in ch_list if ch in erp_E.ch_names and ch not in erp_E.info['bads']]
    if not good_ch_list:
        continue

    ch_indices = [erp_E.ch_names.index(ch) for ch in good_ch_list]

    # Average across channels in this region, ignoring NaNs
    erp_E_region = np.nanmean(erp_E.data[ch_indices, :], axis=0)
    erp_N_region = np.nanmean(erp_N.data[ch_indices, :], axis=0)

    # Plot ERP waveforms
    ax.plot(erp_E.times * 1000, erp_E_region * 1e6, color='black', label='Normal Context', linewidth=1.2)
    ax.plot(erp_E.times * 1000, erp_N_region * 1e6, color='gray', label='Slowed Context', linewidth=1.2)

    ax.set_title(region, fontsize=10)
    ax.axvline(0, color='k', linestyle='--', linewidth=0.5)
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
    ax.set_xlim(-100, 400)
    ax.set_ylim(-30, 30)
    if i == 0:
        ax.legend(loc='upper left', fontsize=8, frameon=False)

# Global axis labels
axs[-1, 1].set_xlabel('Time (ms)', fontsize=12)
axs[1, 0].set_ylabel('Amplitude (μV)', fontsize=12)
plt.suptitle('ERP Waveforms in 9 Scalp Regions (Normalized to μV)', fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

